# Phase 2 Walkthrough: DeepAR (LSTM) & Comparative Evaluation

This notebook demonstrates the Deep Learning component (Phase 2) of Project 5: **Inventory Demand Forecasting via Kalman Filtering and Global DeepAR Models**.

Before running this notebook, ensure you have executed the backtest:
```bash
python3 run_phase2.py
```
This generates `reports/predictions_phase2.csv` and evaluation metrics across all four models:
1. **Seasonal Naive** (baseline)
2. **Kalman Filter** (local state tracking with weekday adjustments)
3. **GBDT** (global gradient boosted trees with lag & calendar features)
4. **DeepAR (LSTM)** (global sequence model with learned uncertainty)

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

with open(project_root / 'config.yaml') as f:
    config = yaml.safe_load(f)

predictions_p2 = pd.read_csv(project_root / 'reports' / 'predictions_phase2.csv')
print(f"Total prediction rows: {len(predictions_p2):,}")
print(f"Models evaluated: {predictions_p2['model'].unique().tolist()}")

## 1. Overall Performance Comparison (Scored Days)

Let us inspect the overall performance across all models on held-out scored periods.

In [ ]:
overall_path = project_root / 'reports' / 'metrics_overall_p2.csv'
if overall_path.exists():
    df_overall = pd.read_csv(overall_path)
    display(df_overall.round(3))
else:
    from report_phase2 import score_table_p2
    scored = predictions_p2[predictions_p2['is_scored'] == 1]
    df_overall = score_table_p2(scored, config['backtest']['quantiles'])
    display(df_overall.round(3))

## 2. Forecast Horizon Degradation: GBDT vs. DeepAR

Recursive forecasting in GBDT compounds error at lead steps. DeepAR autoregressively decodes with latent states.

In [ ]:
from report_phase2 import score_table_p2
scored = predictions_p2[predictions_p2['is_scored'] == 1]
by_step = score_table_p2(scored, config['backtest']['quantiles'], group_column='step')

fig, ax = plt.subplots(figsize=(10, 4.5))
for model, color in [('GBDT', '#1baf7a'), ('DeepAR', '#9b59b6'), ('KalmanFilter', '#eb6834')]:
    subset = by_step[by_step['model'] == model].sort_values('step')
    ax.plot(subset['step'], subset['MAE'], marker='o', label=model, color=color, linewidth=2)

ax.set_title("MAE across Forecast Lead Steps (1 to 14 days)")
ax.set_xlabel("Forecast Step (Days Ahead)")
ax.set_ylabel("Mean Absolute Error (Units)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 3. Cold Start Performance

Comparison on items with less than 90 days of history at forecast origin.

In [ ]:
cold_summary = scored.groupby(['model', 'is_cold_start']).agg(
    actual_mean=('actual', 'mean'),
    point_mean=('point', 'mean'),
    count=('actual', 'count')
).reset_index()
cold_summary['is_cold_start'] = cold_summary['is_cold_start'].map({0: 'Established', 1: 'Cold Start'})
cold_summary